Requirement

Read data from flightt_time and dataframe as below
      1. Rename fl_date to dep_date
      2. Compute arr_date
      3. Following fields to represent full timestamp
        1. crs_dep_time
        2. dep_time
        3. crs_arr_time
        4. arr_time


In [0]:
%sql
select fl_date,crs_dep_time,dep_time
crs_arr_time,arr_time from dev_catalog.spark_db.flight_time

Can We do it using select() or selectExpr() transformations?

```
OPTION 2: select() (PySpark style – CLEAR & BEST PRACTICE)
✔ PySpark API
✔ Type-safe
✔ Easier to debug
✔ Production-friendly
```

```
OPTION 1: selectExpr() (SQL style – SAFE way)
✔ SQL style
✔ selectExpr() transformation
✔ No alias dependency issue 
     1. Example(Wrong Way)
     df.selectExpr(
    "fl_date as dep_date",
    "dep_date + dep_time as new_time"

    2. Example (Correct Way)
    df.selectExpr(
    "fl_date as dep_date",
    "fl_date + dep_time as dep_time_new"
)

##     Ye kyun galat hai?
##     dep_date sirf alias hai
##     Spark left-to-right execute nahi karta
##     Wo bolta hai:
##     “dep_date naam ka column to schema me hai hi nahi”
## Spark ka rule (important)
## Aliases created in the same select are NOT visible to other expressions in that select

)
```


````
| Method         | Style         | When to use               |
| -------------- | ------------- | ------------------------- |
| `selectExpr()` | SQL style     | SQL comfort, short logic  |
| `select()`     | PySpark style | Complex logic, production |
```

In [0]:
flight_time_df = spark.read.table("dev_catalog.spark_db.flight_time")
flight_time_new_df = (
    flight_time_df.selectExpr(
        "fl_date as dep_date"
    )
)
flight_time_new_df.display()

```
## One-line answer (interview ready)

withColumnRenamed only renames a column while keeping all other columns intact, whereas selectExpr selects and renames columns and drops the rest unless explicitly included.
```
```
## Assume ye ORIGINAL DataFrame hai
+-------+--------+-------+
|fl_date|dep_time|carrier|
+-------+--------+-------+
|2023-01-01|  0800 |  AA  |
+-------+--------+-------+

<!-- df1 = sample_df.withColumnRenamed("fl_date", "dep_date")
df1.show() -->
## Output
+--------+--------+-------+
|dep_date|dep_time|carrier|
+--------+--------+-------+
|2023-01-01|  0800 |  AA  |
+--------+--------+-------+
fl_date → dep_date ✅
dep_time → same ✅
carrier → same ✅
👉 Koi column drop nahi hua


## selectExpr()
<!-- df2 = sample_df.selectExpr("fl_date as dep_date")
df2.show() -->
## Output
+--------+
|dep_date|
+--------+
|2023-01-01|
+--------+
fl_date → dep_date ✅
❌ dep_time → DROP
❌ carrier → DROP
👉 Baaki sab columns gayab


## Agar tum chahte ho sab rahein (selectExpr ke saath)
df3 = sample_df.selectExpr(
    "fl_date as dep_date",
    "dep_time",
    "carrier"
)
df3.show()

```

In [0]:
flight_time_df = spark.read.table("dev_catalog.spark_db.flight_time")
flight_time_neww_df = (
    flight_time_df.withColumnRenamed(
        "fl_date", "dep_date"
    )
)
flight_time_neww_df.display()

## Using selectExpr spark method
```
👉 selectExpr() pure PySpark ka method nahi hai,
ye PySpark DataFrame me Spark SQL expressions execute karne ke liye wrapper hai.
```
```
👉 select() pure PySpark DataFrame method hai (SQL nahi).
select() PySpark DataFrame method hai jo columns ko choose/transform karne ke kaam aata hai.
```

In [0]:
flight_time_df = spark.read.table("dev_catalog.spark_db.flight_time")

# Alias dependency
# One-line interview answer
# Aliases created in the same selectExpr cannot be referenced by other expressions in that select; they must be repeated or created in prior steps.

# ```
# Detail me kyun chala? (REAL reasons)
# ✅ Reason 1: Databricks SQL optimizer (Catalyst) ne alias ko expand kar diya


# Final one-line truth

# Tumhara code isliye chala kyunki Databricks optimizer ne alias ko internally resolve kar diya, lekin ye reliable behavior nahi hai aur production me avoid karna chahiye.

# ```

flight_time_3_df = (flight_time_df.selectExpr(
    "fl_date as dep_date",
    "to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date",
        "dep_date + crs_dep_time as crs_dep_time",
        "dep_date + dep_time as dep_time",
        "arr_date + crs_arr_time as crs_arr_time",
        "arr_date + arr_time as arr_time",
        "op_carrier"
)
)
flight_time_3_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()


# Spark ka strict rule (yaad rakh lo)

# Aliases created in the same select / selectExpr are NOT visible to other expressions in that select
# flight_time_3_df = (flight_time_df.selectExpr(
#     "fl_date as dep_date",
#     "to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date",
#         "dep_date + crs_dep_time as crs_dep_time",
#         "dep_date + dep_time as dep_time",
#         "arr_date + crs_arr_time as crs_arr_time",
#         "arr_date + arr_time as arr_time",
#         "op_carrier"
# )
# )

## Using Select() Pyspark method
## selct use alias
## where as selectExpr use as

In [0]:
from pyspark.sql.functions import col, to_date

flight_time_4_df = (
    flight_time_df
    .select(
        col("fl_date").alias("dep_date"),
        to_date(col("fl_date") + col("dep_time") + col("wheels_on") + col("taxi_in"))
            .alias("arr_date"),
        (col("fl_date") + col("crs_dep_time")).alias("crs_dep_time"),
        (col("fl_date") + col("dep_time")).alias("dep_time"),
        (
            to_date(col("fl_date") + col("dep_time") + col("wheels_on") + col("taxi_in"))
            + col("crs_arr_time")
        ).alias("crs_arr_time"),
        (
            to_date(col("fl_date") + col("dep_time") + col("wheels_on") + col("taxi_in"))
            + col("arr_time")
        ).alias("arr_time"),
        col("op_carrier")
    )
)
flight_time_4_df.display()

### Learn Hoo to create file from python

In [0]:
import reportlab

In [0]:
%pip install reportlab

In [0]:
%restart_python


In [0]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

file_path = "/Volumes/dev_catalog/spark_db/datasets/spark_programming/data/pyspark_short_notes.pdf"
doc = SimpleDocTemplate(file_path)
styles = getSampleStyleSheet()
content = []

summary_text = """
PySpark & Databricks Short Notes

1. cast() vs try_cast()
cast = strict, try_cast = safe (returns NULL)

2. coalesce()
Returns first non-null value

3. expr()
Used to write Spark SQL expressions

4. select vs selectExpr
select = PySpark style
selectExpr = SQL style

5. Alias Dependency
Do not reuse aliases in same select/selectExpr

6. withColumnRenamed
Only renames column, does not drop others

7. StructType
Defines schema with full control

8. Data Types
StringType, IntegerType, DoubleType, DateType, ArrayType, MapType, StructType

9. case when
SQL conditional logic, always ends with END

10. uuid4 + lit
Used to generate batch_id

Golden Rule:
Prefer safe, step-by-step transformations in production.
"""

content.append(
    Paragraph(summary_text.replace("\n", "<br/>"), styles["Normal"])
)

doc.build(content)

file_path


Can we do it using withColumn() or withColumns()?

In [0]:
from pyspark.sql.functions import expr

flight_time_6_df = (
  flight_time_df .withColumnRenamed("fl_date", "dep_date")
      .withColumn("arr_date", expr("to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date"))
      .withColumns({
        "crs_dep_time": expr("dep_date + crs_dep_time"),
        "dep_time": expr("dep_date + dep_time"),
        "crs_arr_time": expr("arr_date + crs_arr_time"),
        "arr_time": expr("arr_date + arr_time"),
      })
)

flight_time_6_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()

3. Alternative approach to write expressions
   Why to use it?
-    It gives access to column functions

In [0]:
from pyspark.sql.functions import to_date, col

flight_time_9_df = (
  flight_time_df.withColumnRenamed("fl_date", "dep_date")
      .withColumn("arr_date", to_date(col("dep_date") + col("dep_time") + col("wheels_on") + col("taxi_in")))
      .withColumns({
        "crs_dep_time": col("dep_date") + col("crs_dep_time"),
        "dep_time": col("dep_date") + col("dep_time"),
        "crs_arr_time": col("arr_date") + col("crs_arr_time"),
        "arr_time": col("arr_date") + col("arr_time"),
      })
)

flight_time_9_df.where((col("op_carrier_fl_num") == 1451) & (col("dep_date") == '2000-01-01')).display()